In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [14]:
# !pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [11]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [12]:
docs = [doc1 , doc2 , doc3 , doc4 , doc5]

In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1467.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Create the Vector Store
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='chroma_db',
    collection_name='sample'
)

In [16]:
# Add Documents
vector_store.add_documents(docs)

['8492dde7-79a8-46d9-bcf6-4aedd678ce1d',
 '8cb01b07-a4ef-4805-a015-770f2786201a',
 '9e7760f3-6524-4b35-94ff-65cc5d300a43',
 'd31d6b3b-f11c-497e-81c6-4e06de4ae2fa',
 '1f5635e0-3282-40db-922a-ae1f7982e39d']

In [ ]:
# View the Documents
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['8492dde7-79a8-46d9-bcf6-4aedd678ce1d',
  '8cb01b07-a4ef-4805-a015-770f2786201a',
  '9e7760f3-6524-4b35-94ff-65cc5d300a43',
  'd31d6b3b-f11c-497e-81c6-4e06de4ae2fa',
  '1f5635e0-3282-40db-922a-ae1f7982e39d'],
 'embeddings': array([[ 0.00994723,  0.06914338, -0.05147114, ..., -0.0354334 ,
          0.01284808,  0.01248293],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [19]:
# search the Documents
vector_store.similarity_search(
    query="Who among these are bowler?",
    k = 1
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [21]:
# search with similarity score
vector_store.similarity_search_with_score(
    query="Who among these are bowler?",
    k = 2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9650976657867432),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.1562845706939697)]

In [22]:
# Filtering through meta_data
vector_store.similarity_search_with_score(
    query='',
    filter={"team":"Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436007499694824),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937089920044)]

In [25]:
# ADD/UPDATE the Document
updated_doc1 = Document(
    page_content="Virat Kohli, the former Captain of Royal Challengers Bangalore (RCB), is renowed for his aggressive leadership and consistency",
    metadata = {"team":"Royal Challengers Bangalore"}
)
vector_store.update_document(document_id = "8492dde7-79a8-46d9-bcf6-4aedd678ce1d",document=updated_doc1)

In [26]:
# View Documents
vector_store.get(include=["embeddings","documents","metadatas"])

{'ids': ['8492dde7-79a8-46d9-bcf6-4aedd678ce1d',
  '8cb01b07-a4ef-4805-a015-770f2786201a',
  '9e7760f3-6524-4b35-94ff-65cc5d300a43',
  'd31d6b3b-f11c-497e-81c6-4e06de4ae2fa',
  '1f5635e0-3282-40db-922a-ae1f7982e39d'],
 'embeddings': array([[ 0.02133356,  0.02585978, -0.05319259, ..., -0.05650486,
          0.01333693, -0.0627773 ],
        [ 0.00127743,  0.03129851, -0.02375379, ..., -0.00518362,
         -0.03280614,  0.02737717],
        [-0.10265917,  0.02650812,  0.02271503, ..., -0.03359742,
         -0.07984945, -0.01507708],
        [ 0.02123393, -0.02468549, -0.04494374, ..., -0.10995809,
          0.0057256 ,  0.09915379],
        [ 0.01873982,  0.04382847, -0.04304255, ..., -0.07801619,
         -0.07840681, -0.00304189]]),
 'documents': ['Virat Kohli, the former Captain of Royal Challengers Bangalore (RCB), is renowed for his aggressive leadership and consistency',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's know